In [ ]:
# In[1]:

import pandas as pd

# Load CSV files
metric_df = pd.read_csv("metric.csv")
trace_df = pd.read_csv("trace.csv")
log_df = pd.read_csv("log.csv")
error_df = pd.read_csv("error_logs.csv")

# Parse timestamps to UTC datetimes (new column 'ts')
for df in (metric_df, trace_df, log_df, error_df):
    df['ts'] = pd.to_datetime(df['timestamp'], unit='s', utc=True)

# --- metric.csv summaries ---
metric_summary = pd.DataFrame([{
    'file': 'metric.csv',
    'total_rows': len(metric_df),
    'min_ts_utc': metric_df['ts'].min(),
    'max_ts_utc': metric_df['ts'].max()
}])

metric_cmdb_top = metric_df['cmdb_id'].value_counts().head(20).rename_axis('cmdb_id').reset_index(name='count')
metric_kpi_top = metric_df['kpi_name'].value_counts().head(20).rename_axis('kpi_name').reset_index(name='count')

metric_pairs_top = (
    metric_df.groupby(['cmdb_id', 'kpi_name'])
    .size()
    .reset_index(name='count')
    .sort_values('count', ascending=False)
    .head(20)
)

# --- trace.csv summaries ---
trace_summary = pd.DataFrame([{
    'file': 'trace.csv',
    'total_rows': len(trace_df),
    'min_ts_utc': trace_df['ts'].min(),
    'max_ts_utc': trace_df['ts'].max()
}])

trace_cmdb_top = trace_df['cmdb_id'].value_counts().head(20).rename_axis('cmdb_id').reset_index(name='count')
trace_name_top = trace_df['trace_name'].value_counts().head(20).rename_axis('trace_name').reset_index(name='count')

# --- log.csv summaries ---
log_summary = pd.DataFrame([{
    'file': 'log.csv',
    'total_rows': len(log_df),
    'min_ts_utc': log_df['ts'].min(),
    'max_ts_utc': log_df['ts'].max()
}])

log_cmdb_top = log_df['cmdb_id'].value_counts().head(20).rename_axis('cmdb_id').reset_index(name='count')
log_name_top = log_df['log_name'].value_counts().head(20).rename_axis('log_name').reset_index(name='count')

# --- error_logs.csv summaries ---
# Trim messages to 100 chars for counting/display
error_df['msg_trim'] = error_df['message'].astype(str).str.slice(0, 100)

error_summary = pd.DataFrame([{
    'file': 'error_logs.csv',
    'total_rows': len(error_df),
    'min_ts_utc': error_df['ts'].min(),
    'max_ts_utc': error_df['ts'].max()
}])

error_cmdb_top = error_df['cmdb_id'].value_counts().head(20).rename_axis('cmdb_id').reset_index(name='count')
error_message_top = error_df['msg_trim'].value_counts().head(20).rename_axis('message_trim_100').reset_index(name='count')

# Display compact summaries (only selected compact outputs)
metric_summary, metric_cmdb_top, metric_kpi_top, metric_pairs_top, trace_summary, trace_cmdb_top, trace_name_top, log_summary, log_cmdb_top, log_name_top, error_summary, error_cmdb_top, error_message_top

```
Out[1]:
```
Summary of the telemetry (concise observations):

Overall timeframe
- Most telemetry spans 2024-01-17 19:06:00+00:00 through 2024-01-17 19:30:00+00:00.
- error_logs are narrower: 2024-01-17 19:20:31+00:00 → 2024-01-17 19:29:49+00:00.

metric.csv (1834 rows)
- Top components (by row count): adservice, cartservice, checkoutservice, emailservice, paymentservice, frontend, recommendationservice, productcatalogservice, currencyservice, shippingservice.
- Top KPI names: cpu, workload, socket, mem, latency-50, latency-90, diskio, error.
- Top component-KPI series (examples): adservice/cpu, adservice/diskio, cartservice/cpu, checkoutservice/cpu, etc. (multiple services have cpu, mem, socket, workload and latency series).

trace.csv (3656 rows)
- Top components: checkoutservice, root, frontendservice, recommendationservice, productcatalogservice, currencyservice, emailservice, paymentservice.
- Top trace features: many "trace.from_root.*" series and "trace.from_frontendservice.*" series; also traces to productcatalogservice and recommendationservice (duration_mean, duration_p95, error_rate, row_count).

log.csv (462 rows)
- Top components: cartservice, frontend, adservice, shippingservice, currencyservice, recommendationservice, checkoutservice, paymentservice, emailservice.
- Top log fields: log.error_count, log.row_count.

error_logs.csv (523 rows)
- All error rows are for cmdb_id = frontend (523).
- Top trimmed messages (first 100 chars):
  - "failed to get product recommendations" — 374 occurrences
  - "request error" — 149 occurrences
- Error timestamps are concentrated between 19:20:31 and 19:29:49 UTC.

Key takeaway (observation, not a definitive root-cause claim)
- The error logs are dominated by frontend errors reporting "failed to get product recommendations" during 19:20–19:29 UTC. Trace telemetry shows activity and traces involving recommendationservice and productcatalogservice, and metrics include latency/socket/cpu series across services. These facts point to a problem in the frontend’s interactions with the recommendation path (frontend → recommendationservice) during that interval and merit focused investigation of the recommendationservice and the frontend→recommendation network/requests.

The original code execution output of IPython Kernel is also provided below for reference:

(         file  total_rows                min_ts_utc                max_ts_utc
0  metric.csv        1834 2024-01-17 19:06:00+00:00 2024-01-17 19:30:00+00:00,                   cmdb_id  count
0               adservice    175
1             cartservice    175
2         checkoutservice    175
3            emailservice    175
4          paymentservice    175
5                frontend    175
6   recommendationservice    175
7   productcatalogservice    159
8         currencyservice    150
9         shippingservice    150
10                  redis    100
11      frontend-external     50,      kpi_name  count
0         cpu    275
1    workload    275
2      socket    275
3         mem    275
4  latency-50    250
5  latency-90    250
6      diskio    175
7       error     59,             cmdb_id    kpi_name  count
0         adservice         cpu     25
1         adservice      diskio     25
2         adservice  latency-50     25
3         adservice  latency-90     25
4         adservice         mem     25
5         adservice      socket     25
6         adservice    workload     25
7       cartservice         cpu     25
8       cartservice      diskio     25
9       cartservice  latency-50     25
10      cartservice  latency-90     25
11      cartservice         mem     25
12      cartservice      socket     25
13      cartservice    workload     25
14  checkoutservice         cpu     25
15  checkoutservice      diskio     25
16  checkoutservice  latency-50     25
17  checkoutservice  latency-90     25
18  checkoutservice         mem     25
19  checkoutservice      socket     25,         file  total_rows                min_ts_utc                max_ts_utc
0  trace.csv        3656 2024-01-17 19:06:00+00:00 2024-01-17 19:30:00+00:00,                  cmdb_id  count
0        checkoutservice    744
1                   root    700
2        frontendservice    672
3  recommendationservice    484
4  productcatalogservice    384
5        currencyservice    288
6           emailservice    192
7         paymentservice    192,                                       trace_name  count
0                  trace.from_root.duration_mean    175
1                   trace.from_root.duration_p95    175
2                     trace.from_root.error_rate    175
3                      trace.from_root.row_count    175
4           trace.from_frontendservice.row_count    119
5          trace.from_frontendservice.error_rate    119
6        trace.from_frontendservice.duration_p95    119
7       trace.from_frontendservice.duration_mean    119
8           trace.from_checkoutservice.row_count    115
9          trace.from_checkoutservice.error_rate    115
10       trace.from_checkoutservice.duration_p95    115
11      trace.from_checkoutservice.duration_mean    115
12      trace.to_productcatalogservice.row_count     96
13     trace.to_productcatalogservice.error_rate     96
14   trace.to_productcatalogservice.duration_p95     96
15  trace.to_productcatalogservice.duration_mean     96
16  trace.to_recommendationservice.duration_mean     73
17   trace.to_recommendationservice.duration_p95     73
18     trace.to_recommendationservice.error_rate     73
19      trace.to_recommendationservice.row_count     73,       file  total_rows                min_ts_utc                max_ts_utc
0  log.csv         462 2024-01-17 19:06:00+00:00 2024-01-17 19:30:00+00:00,                   cmdb_id  count
0             cartservice     50
1                frontend     50
2               adservice     48
3         shippingservice     48
4         currencyservice     48
5   recommendationservice     48
6         checkoutservice     46
7          paymentservice     46
8            emailservice     46
9   productcatalogservice     22
10                  redis     10,           log_name  count
0  log.error_count    231
1    log.row_count    231,              file  total_rows                min_ts_utc                max_ts_utc
0  error_logs.csv         523 2024-01-17 19:20:31+00:00 2024-01-17 19:29:49+00:00,     cmdb_id  count
0  frontend    523,                         message_trim_100  count
0  failed to get product recommendations    374
1                          request error    149)```
```

In [ ]:
# In[2]:

# Reuse existing metric_df and its 'ts' column from earlier state
import pandas as pd

# Ensure metric_df is present
try:
    metric_df
except NameError:
    metric_df = pd.read_csv("metric.csv")
    metric_df['ts'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True)

# 1) Compute global P95 per (cmdb_id, kpi_name) using the full metric_df
group_p95 = (
    metric_df
    .groupby(['cmdb_id', 'kpi_name'])['value']
    .quantile(0.95)
    .reset_index(name='global_p95')
)

# 2) Define incident window (UTC)
start = pd.to_datetime("2024-01-17 19:03:34", utc=True)
end   = pd.to_datetime("2024-01-17 19:33:34", utc=True)

# Candidate services
candidates = [
    "checkoutservice",
    "currencyservice",
    "emailservice",
    "productcatalogservice",
    "recommendationservice"
]

# 3) Filter metric_df to the incident window and candidate services
window_df = metric_df[
    (metric_df['ts'] >= start) &
    (metric_df['ts'] <= end) &
    (metric_df['cmdb_id'].isin(candidates))
].copy()

# If empty, produce empty result table
if window_df.shape[0] == 0:
    result_df = pd.DataFrame(columns=[
        'cmdb_id','kpi_name','total_points_in_window','count_above_P95',
        'earliest_timestamp_above_P95','max_value_in_window','global_P95_threshold'
    ])
else:
    # 4) Attach global_p95 to windowed rows
    window_df = window_df.merge(group_p95, on=['cmdb_id','kpi_name'], how='left')

    # 5) Mark points above global P95 (strictly greater than P95)
    window_df['above'] = window_df['value'] > window_df['global_p95']

    # 6) Aggregate per (cmdb_id, kpi_name)
    grp = window_df.groupby(['cmdb_id','kpi_name'])

    total_points = grp.size().rename('total_points_in_window')
    count_above = grp['above'].sum().astype(int).rename('count_above_P95')
    max_value = grp['value'].max().rename('max_value_in_window')
    global_p95_first = grp['global_p95'].first().rename('global_P95_threshold')

    # earliest timestamp where above==True
    earliest_above = (
        window_df[window_df['above']]
        .groupby(['cmdb_id','kpi_name'])['ts']
        .min()
        .rename('earliest_timestamp_above_P95')
    )

    # Combine aggregates
    result_df = pd.concat([total_points, count_above, earliest_above, max_value, global_p95_first], axis=1).reset_index()

    # Ensure column for earliest exists for groups with no above (will be NaT)
    if 'earliest_timestamp_above_P95' not in result_df.columns:
        result_df['earliest_timestamp_above_P95'] = pd.NaT

    # Only include rows where total_points_in_window > 0 (window_df grouping already ensures this)
    result_df = result_df[result_df['total_points_in_window'] > 0]

    # Sort by count_above_P95 desc then earliest_timestamp_above_P95 asc (NaT sorts last)
    result_df = result_df.sort_values(
        by=['count_above_P95', 'earliest_timestamp_above_P95'],
        ascending=[False, True]
    )

# Keep output compact (limit 50 rows)
result_df = result_df.head(50)

# Display the result table
result_df

```
Out[2]:
```
# Summarize the previously computed result_df into a concise plain-English summary string
try:
    result_df
except NameError:
    import pandas as pd
    result_df = pd.read_csv("metric.csv")  # fallback, but result_df should exist from prior step

# Select only series with any points in window (already ensured) and those with >0 above-P95
anomalies = result_df[result_df['count_above_P95'] > 0].copy()

# Prepare a compact top list (limit 10) of the anomaly rows with key fields
top_list = anomalies.sort_values(['count_above_P95', 'earliest_timestamp_above_P95'], ascending=[False, True]).head(10)

lines = []
lines.append(f"Summary for candidate services within 2024-01-17 19:03:34 → 19:33:34 UTC:")
lines.append(f"- Total (cmdb_id, kpi_name) series returned: {len(result_df)}")
lines.append(f"- Series with values above their global P95 in the window: {len(anomalies)}")
lines.append(f"- Maximum count_above_P95 observed: {int(anomalies['count_above_P95'].max()) if not anomalies.empty else 0}")
lines.append("")
lines.append("Top anomaly series (up to 10) with fields: cmdb_id | kpi_name | count_above_P95 | earliest_timestamp_above_P95 | max_value_in_window | global_P95_threshold")
for _, r in top_list.iterrows():
    et = r['earliest_timestamp_above_P95']
    et_s = str(et) if pd.notna(et) else "NA"
    lines.append(f"- {r['cmdb_id']} | {r['kpi_name']} | {int(r['count_above_P95'])} | {et_s} | {r['max_value_in_window']} | {r['global_P95_threshold']}")

lines.append("")
lines.append("Observations:")
lines.append("- Several series show a small number (2) of values exceeding their global P95 during the incident window.")
lines.append("- Affected series include workload and cpu KPIs for checkoutservice, currencyservice, and emailservice (among others).")
lines.append("- Earliest exceedances occur around 2024-01-17 19:06:00+00:00 (one checkout cpu at 19:07).")
lines.append("- No series shows a large sustained flood of values above P95 in this window (counts above P95 are small).")
lines.append("")
lines.append("Recommendation: Investigate the listed series (especially workload and cpu for checkoutservice, currencyservice, emailservice) and correlate with traces/errors in the same timestamps to find root cause.")

summary = "\n".join(lines)
summary, len(result_df), len(anomalies)

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id  kpi_name  total_points_in_window  count_above_P95 earliest_timestamp_above_P95  max_value_in_window  global_P95_threshold
6         checkoutservice  workload                      25                2    2024-01-17 19:06:00+00:00             1.218154              1.078487
12        currencyservice  workload                      25                2    2024-01-17 19:06:00+00:00            82.039000             81.307981
13           emailservice       cpu                      25                2    2024-01-17 19:06:00+00:00             0.347522              0.331314
19           emailservice  workload                      25                2    2024-01-17 19:06:00+00:00             1.251115              1.080017
0         checkoutservice       cpu                      25                2    2024-01-17 19:07:00+00:00             0.536755              0.448905
..                    ...       ...                     ...              ...                          ...                  ...                   ...
5         checkoutservice    socket                      25                0                          NaT             9.000000              9.000000
11        currencyservice    socket                      25                0                          NaT             4.000000              4.000000
14           emailservice    diskio                      25                0                          NaT             0.000000              0.000000
18           emailservice    socket                      25                0                          NaT             3.000000              3.000000
28  recommendationservice    diskio                      25                0                          NaT             0.000000              0.000000

[34 rows x 7 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[3]:

import pandas as pd

# Reuse trace_df if available, otherwise load
try:
    trace_df
except NameError:
    trace_df = pd.read_csv("trace.csv")
    trace_df['ts'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True)

# Ensure ts parsed
if 'ts' not in trace_df.columns:
    trace_df['ts'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True)

# 1) Compute global P95 and P05 per (cmdb_id, trace_name) using full trace_df
thr = (
    trace_df
    .groupby(['cmdb_id', 'trace_name'])['value']
    .agg(
        global_p95=lambda s: s.quantile(0.95),
        global_p05=lambda s: s.quantile(0.05)
    )
    .reset_index()
)

# 2) Define incident window (UTC)
start = pd.to_datetime("2024-01-17 19:03:34", utc=True)
end   = pd.to_datetime("2024-01-17 19:33:34", utc=True)

# 3) Filter trace_df to window and to trace_name patterns
patterns = ['to_recommendationservice', 'to_productcatalogservice', 'from_frontendservice']
pattern_regex = '|'.join(patterns)

window_trace = trace_df[
    (trace_df['ts'] >= start) &
    (trace_df['ts'] <= end) &
    (trace_df['trace_name'].str.contains(pattern_regex))
].copy()

# If no rows, return empty structured dataframe
if window_trace.shape[0] == 0:
    result_trace_anomalies = pd.DataFrame(columns=[
        'cmdb_id','trace_name','total_points_in_window','metric_type',
        'count_above_global_P95','count_below_global_P05','earliest_timestamp_of_anomaly',
        'max_value_in_window','min_value_in_window','global_P95_threshold','global_P05_threshold'
    ])
else:
    # 4) Merge thresholds
    window_trace = window_trace.merge(thr, on=['cmdb_id','trace_name'], how='left')

    # 5) Determine metric_type
    def mt(name):
        if 'error_rate' in name:
            return 'error_rate'
        if 'row_count' in name:
            return 'row_count'
        if 'duration' in name:
            return 'duration'
        return 'other'
    window_trace['metric_type'] = window_trace['trace_name'].apply(mt)

    # 6) Flag anomalies per row
    window_trace['above_p95'] = window_trace['value'] > window_trace['global_p95']
    window_trace['below_p05'] = window_trace['value'] < window_trace['global_p05']

    # 7) Aggregate per (cmdb_id, trace_name)
    grp = window_trace.groupby(['cmdb_id','trace_name','metric_type'])

    total_points = grp.size().rename('total_points_in_window')
    count_above = grp['above_p95'].sum().astype(int).rename('count_above_global_P95')
    count_below = grp['below_p05'].sum().astype(int).rename('count_below_global_P05')
    max_val = grp['value'].max().rename('max_value_in_window')
    min_val = grp['value'].min().rename('min_value_in_window')
    gp95 = grp['global_p95'].first().rename('global_P95_threshold')
    gp05 = grp['global_p05'].first().rename('global_P05_threshold')

    # earliest timestamp of anomaly: depends on metric_type
    # For error_rate/duration use above_p95, for row_count use below_p05
    def earliest_anom(df):
        mt = df.name[2]  # groupby grouping (cmdb_id, trace_name, metric_type)
        if mt in ('error_rate','duration'):
            sel = df[df['above_p95']]
        elif mt == 'row_count':
            sel = df[df['below_p05']]
        else:
            sel = df.iloc[0:0]
        if sel.shape[0] == 0:
            return pd.NaT
        return sel['ts'].min()

    earliest = (
        window_trace
        .groupby(['cmdb_id','trace_name','metric_type'])
        .apply(lambda d: (d[d['above_p95']]['ts'].min() if d.name[2] in ('error_rate','duration') else (d[d['below_p05']]['ts'].min() if d.name[2]=='row_count' else pd.NaT)))
        .rename('earliest_timestamp_of_anomaly')
    )

    # Combine aggregates
    agg = pd.concat([total_points, count_above, count_below, earliest, max_val, min_val, gp95, gp05], axis=1).reset_index()

    # 8) Compute primary count for sorting: above for error_rate/duration, below for row_count
    agg['primary_count'] = agg.apply(lambda r: r['count_above_global_P95'] if r['metric_type'] in ('error_rate','duration') else r['count_below_global_P05'], axis=1)

    # 9) Keep only requested columns and sort
    result_trace_anomalies = agg[
        ['cmdb_id','trace_name','total_points_in_window','metric_type',
         'count_above_global_P95','count_below_global_P05','earliest_timestamp_of_anomaly',
         'max_value_in_window','min_value_in_window','global_P95_threshold','global_P05_threshold','primary_count']
    ].sort_values(by=['primary_count','earliest_timestamp_of_anomaly'], ascending=[False, True]).drop(columns=['primary_count'])

    # 10) Limit to 20 rows
    result_trace_anomalies = result_trace_anomalies.head(20)

# Display compact result
result_trace_anomalies

```
Out[3]:
```
Summary of trace-anomaly aggregates (incident window 2024-01-17 19:03:34 → 19:33:34 UTC):

Key findings
- Multiple trace series involving recommendationservice, productcatalogservice, and calls from the frontend show anomalies (mostly 2 points above their global P95 or 2 points below their global P05).
- Anomalous timestamps cluster around two times: ~19:06–19:07 UTC and ~19:21 UTC.
- Types of anomalies observed:
  - duration (many series): elevated durations above global P95.
  - row_count (some series): decreased row_count below global P05.
  - error_rate (frontend): increased error_rate above global P95.

Notable series (representative examples)
- recommendationservice / trace.to_productcatalogservice.duration_mean — 24 points in window; 2 points above global P95; earliest above at 2024-01-17 19:06:00 UTC.
- root / trace.to_recommendationservice.row_count — 25 points; 2 points below global P05; earliest at 2024-01-17 19:06:00 UTC.
- currencyservice / trace.from_frontendservice.duration_mean — 24 points; 2 above P95; earliest at 2024-01-17 19:07:00 UTC.
- checkoutservice / trace.from_frontendservice.duration_mean and duration_p95 — 23 points each; 2 above P95; earliest anomalies at ~2024-01-17 19:21:00 UTC.
- frontendservice / trace.to_recommendationservice.duration_mean and duration_p95 — 24 points; 2 above P95; earliest at 2024-01-17 19:21:00 UTC.
- recommendationservice / trace.from_frontendservice.duration_mean and duration_p95 — 24 points; 2 above P95; earliest at 2024-01-17 19:21:00 UTC.
- root / trace.to_productcatalogservice.row_count — 25 points; 2 below P05; earliest at 2024-01-17 19:20:00 UTC.
- currencyservice / trace.from_frontendservice.row_count — 24 points; 2 below P05; earliest at 2024-01-17 19:22:00 UTC.
- frontendservice / trace.from_frontendservice.error_rate — 24 points; 2 above global P95; earliest at 2024-01-17 19:22:00 UTC.

Interpretation / actionable guidance
- The trace anomalies implicate the recommendationservice and productcatalogservice call paths and the frontend→recommendation/productcatalog interactions. Elevated durations and increased frontend error_rate, together with reduced row_counts to recommendation/productcatalog, suggest degraded downstream responsiveness or failed responses in that path.
- Investigate recommendationservice and productcatalogservice around 19:06 and especially around 19:21 UTC. Correlate these trace anomalies with the frontend error logs (which showed "failed to get product recommendations") and metric anomalies to pinpoint whether the issue is a backend slowdown, resource exhaustion, or request failures.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                    trace_name  total_points_in_window metric_type  count_above_global_P95  count_below_global_P05 earliest_timestamp_of_anomaly  max_value_in_window  min_value_in_window  global_P95_threshold  global_P05_threshold
32  recommendationservice  trace.to_productcatalogservice.duration_mean                      24    duration                       2                       2     2024-01-17 19:06:00+00:00             0.000019             0.000011              0.000017              0.000012
47                   root      trace.to_recommendationservice.row_count                      25   row_count                       0                       2     2024-01-17 19:06:00+00:00            24.000000             6.000000             24.000000             10.800000
8         currencyservice      trace.from_frontendservice.duration_mean                      24    duration                       2                       2     2024-01-17 19:07:00+00:00             0.000386             0.000106              0.000331              0.000126
4         checkoutservice  trace.to_productcatalogservice.duration_mean                      23    duration                       2                       2     2024-01-17 19:11:00+00:00             0.000018             0.000012              0.000018              0.000013
9         currencyservice       trace.from_frontendservice.duration_p95                      24    duration                       2                       2     2024-01-17 19:19:00+00:00             0.000256             0.000188              0.000250              0.000190
43                   root      trace.to_productcatalogservice.row_count                      25   row_count                       2                       2     2024-01-17 19:20:00+00:00            34.000000             6.000000             28.800000              6.400000
0         checkoutservice      trace.from_frontendservice.duration_mean                      23    duration                       2                       2     2024-01-17 19:21:00+00:00             1.595279             0.059862              1.518497              0.078931
1         checkoutservice       trace.from_frontendservice.duration_p95                      23    duration                       2                       2     2024-01-17 19:21:00+00:00             5.490024             0.115524              4.289446              0.116416
5         checkoutservice   trace.to_productcatalogservice.duration_p95                      23    duration                       2                       2     2024-01-17 19:21:00+00:00             0.000037             0.000016              0.000036              0.000016
20        frontendservice  trace.to_recommendationservice.duration_mean                      24    duration                       2                       2     2024-01-17 19:21:00+00:00             3.200461             0.003520              1.276002              0.003535
21        frontendservice   trace.to_recommendationservice.duration_p95                      24    duration                       2                       2     2024-01-17 19:21:00+00:00            13.234064             0.004648              6.265396              0.004666
28  recommendationservice      trace.from_frontendservice.duration_mean                      24    duration                       2                       2     2024-01-17 19:21:00+00:00             3.200461             0.003520              1.276002              0.003535
29  recommendationservice       trace.from_frontendservice.duration_p95                      24    duration                       2                       2     2024-01-17 19:21:00+00:00            13.234064             0.004648              6.265396              0.004666
36  recommendationservice  trace.to_recommendationservice.duration_mean                      24    duration                       2                       2     2024-01-17 19:21:00+00:00             3.199655             0.003012              1.275011              0.003023
37  recommendationservice   trace.to_recommendationservice.duration_p95                      24    duration                       2                       2     2024-01-17 19:21:00+00:00            13.233503             0.003951              6.262686              0.004027
44                   root  trace.to_recommendationservice.duration_mean                      25    duration                       2                       2     2024-01-17 19:21:00+00:00             0.000096             0.000041              0.000057              0.000042
45                   root   trace.to_recommendationservice.duration_p95                      25    duration                       2                       1     2024-01-17 19:21:00+00:00             0.000346             0.000049              0.000099              0.000052
11        currencyservice          trace.from_frontendservice.row_count                      24   row_count                       2                       2     2024-01-17 19:22:00+00:00          2357.000000           203.000000           2338.000000            564.400000
13        frontendservice       trace.from_frontendservice.duration_p95                      24    duration                       2                       2     2024-01-17 19:22:00+00:00            23.820295             0.008694             18.241007              0.009375
14        frontendservice         trace.from_frontendservice.error_rate                      24  error_rate                       2                       0     2024-01-17 19:22:00+00:00             0.085694             0.000000              0.043680              0.000000```
```